# Combine Preprocessing Helpers into a Reusable Pipeline

The full implementation lives in [`preprocessing_pipeline.py`](preprocessing_pipeline.py) in the same folder.

## 1 - Setup

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


`preprocessing_pipeline.py` now includes the Gemini call directly, so the notebook setup stays simple. Just load your API key, import the pipeline, and run it.

In [ ]:
%pip install -q google-genai pandas scikit-learn python-dotenv

In [1]:
import pandas as pd
import preprocessing_pipeline as pp

# preprocessing_pipeline.py reads GEMINI_API_KEY from the environment


## 2 - Load the dataset

In [2]:
df = pd.read_csv("../../data/hr_analytics.csv")
df.head()

,Employee ID,age,gender,department,department_code,JobTitle,job_level,Education,MonthlyIncome,monthly_rate,...,satisfaction_score,environment_satisfaction,Attrition,OverTime,distance_from_home,training_hours_last_year,num_companies_worked,manager_rating,work_life_balance,last_promotion_date
0,1001,27,Female,Engineering,ENG-02,Backend Developer,3,High School,8001,9162,...,3 - High,4,No,No,11.0,32.0,1,4.0,Very High,2024-09-06
1,1002,34,Male,Engineering,ENG-02,Data Engineer,2,Master's,8777,10301,...,2 - Medium,2,No,No,26.0,30.0,0,2.0,Low,12/18/2022
2,1003,50,Female,Finance,FIN-05,Controller,4,Master's,14021,18470,...,4 - Very High,3,No,No,17.0,40.0,2,3.0,Medium,2017-12-04
3,1004,31,Male,Marketing,MKT-04,Marketing Analyst,3,Bachelor's,6402,8805,...,4 - Very High,3,No,No,3.0,32.0,0,2.0,High,2024-12-07
4,1005,51,Male,Marketing,MKT-04,SEO Specialist,4,Master's,9277,12125,...,4 - Very High,2,No,No,3.0,25.0,0,1.0,High,06/25/2021


## Full pipeline (clean + encode + split)

In [3]:
result = pp.preprocessing_pipeline(
    raw_path="../../data/hr_analytics.csv",
    target_col="Attrition",
    task="classification",  # or "regression"
)


Loaded: (5030, 25)
Split raw: train=(4024, 25) test=(1006, 25)
Cleaned: train=(4003, 25) missing=0 | test=(1006, 25) missing=0
Encoded: train=(4003, 50) test=(1006, 50)


In [12]:
print(
    result.X_train_enc.shape,
)
print(
    result.X_test_enc.shape,
)
print(
    result.y_train.shape,
)
print(
    result.y_test.shape,
)

(4003, 50)
(1006, 50)
(4003,)
(1006,)


In [13]:
print("Cleaning plan:\n")
print(result.cleaning_plan)

Cleaning plan:

Here are the data quality issues found in the dataset profile:

*   **Column**: Employee ID
    *   **Problem**: Duplicate values exist for an identifier column, which should be unique. The `top_values` show several IDs appearing twice.
    *   **Recommended Fix**: Investigate the source of duplicate Employee IDs. If these represent distinct employees, assign new unique identifiers. If they are the same employee with multiple records, consolidate the records or identify the correct unique ID.

*   **Column**: satisfaction_score
    *   **Problem**: The data type is `str`, but the values represent an ordinal numerical scale (e.g., '3 - High', '4 - Very High'). Storing it as a string prevents direct numerical analysis or proper sorting based on the score.
    *   **Recommended Fix**: Extract the numerical part from the string (e.g., '3' from '3 - High') and convert the column to an integer data type. Optionally, convert to a pandas Categorical type with ordered categories

In [14]:
print("\nEncoding plan:\n")
print(result.encoding_plan)


Encoding plan:

[{'column': 'Employee ID', 'strategy': 'skip', 'reason': 'Identifier column.'}, {'column': 'age', 'strategy': 'scale', 'reason': 'Continuous numeric variable.'}, {'column': 'gender', 'strategy': 'onehot', 'reason': 'Nominal categorical variable with more than 2 unique values.'}, {'column': 'department', 'strategy': 'onehot', 'reason': 'Nominal categorical variable with more than 2 unique values.'}, {'column': 'department_code', 'strategy': 'onehot', 'reason': 'Nominal categorical variable with more than 2 unique values.'}, {'column': 'JobTitle', 'strategy': 'onehot', 'reason': 'Nominal categorical variable with more than 2 unique values.'}, {'column': 'job_level', 'strategy': 'ordinal', 'reason': 'Ordered categorical variable (1, 2, 3, 4, 5).'}, {'column': 'Education', 'strategy': 'ordinal', 'reason': "Ordered categorical variable (High School, Bachelor's, Master's, PhD)."}, {'column': 'MonthlyIncome', 'strategy': 'scale', 'reason': 'Continuous numeric variable.'}, {'c

In [15]:
feature_names = result.encoder.get_feature_names_out()

X_train_df = pd.DataFrame(result.X_train_enc, columns=feature_names)
X_test_df = pd.DataFrame(result.X_test_enc, columns=feature_names)

X_train_df.head()


,scaler__age,scaler__MonthlyIncome,scaler__monthly_rate,scaler__hourly_rate,scaler__daily_rate,scaler__years_at_company,scaler__years_in_role,scaler__years_since_promotion,scaler__distance_from_home,scaler__training_hours_last_year,...,onehot__JobTitle_infrequent_sklearn,onehot__OverTime_No,onehot__OverTime_Yes,ordinal__job_level,ordinal__Education,ordinal__satisfaction_score,ordinal__environment_satisfaction,ordinal__manager_rating,ordinal__work_life_balance,remainder__Employee ID
0,-0.483668,-1.026767,-1.033037,-1.005114,-1.012928,-0.712408,-0.298039,0.114853,-1.029332,-1.302518,...,0.0,1.0,0.0,0.0,1.0,3.0,3.0,3.0,3.0,3223.0
1,-0.132497,-0.803761,-0.987197,-0.798457,-0.815928,-0.866591,-0.754359,-0.589755,-0.702016,0.019764,...,0.0,1.0,0.0,2.0,0.0,3.0,3.0,3.0,3.0,5029.0
2,-0.834838,0.429542,0.399604,0.441484,0.400331,-0.712408,-0.298039,-0.589755,0.443589,0.350335,...,0.0,1.0,0.0,1.0,0.0,1.0,2.0,2.0,0.0,1578.0
3,2.442755,1.182585,1.185974,1.130341,1.042721,2.371252,1.299081,1.171765,-0.702016,0.791096,...,1.0,1.0,0.0,1.0,3.0,3.0,3.0,2.0,3.0,3612.0
4,0.569845,-0.941945,-1.084686,-0.936229,-0.910145,0.212690,-0.069879,0.467157,-0.374700,0.570715,...,0.0,1.0,0.0,0.0,2.0,3.0,3.0,2.0,2.0,2546.0
